# Introduction

This Notebook implements a item-based collaborative filtering recommender system.

# Data preparation

In [1]:
import os
import torch
import pandas as pd
import torch.nn.functional as F

In [2]:
def load_ratings(path):
    data = []

    with open(path, 'r', encoding='latin-1') as f:
        for line in f:
            user_id, movie_id, rating, _ = line.strip().split("::")
            data.append((int(user_id), int(movie_id), float(rating)))

    return data

def load_movies(path):
    movies = {}

    with open(path, 'r', encoding='latin-1') as f:
        for line in f:
            parts = line.strip().split("::")

            movie_id = int(parts[0])
            title = parts[1]

            movies[movie_id - 1] = title  # align with matrix index

    return movies


In [3]:
root_path = "/kaggle/input/datasets/sherinclaudia/movielens"
ratings_path = os.path.join(root_path, "ratings.dat")
movies_path = os.path.join(root_path, "movies.dat")

In [4]:
ratings = load_ratings(ratings_path)
ratings_df = pd.DataFrame(ratings, columns=["user_id", "movie_id", "rating"])

movies = load_movies(movies_path)
movies_df = movies_df = pd.DataFrame.from_dict(movies, orient='index', columns=['title'])
movies_df = movies_df.reset_index()
movies_df.rename(columns={'index': 'movie_id'}, inplace=True)

In [5]:
df = pd.merge(ratings_df, movies_df, on="movie_id")

user_item = df.pivot_table(
    index="user_id",
    columns="title",
    values="rating"
)

user_item_filled = user_item.fillna(0)

ratings = torch.tensor(user_item_filled.values, dtype=torch.float32)

print(f"Ratings shape: {ratings.shape}")

Ratings shape: torch.Size([6040, 3650])


Here, rows are users and columns are movies. For item-based collaborative filtering, however, we compare columns rather than rows. Each movie is represented by the vector of user ratings it received.

In [6]:
# Transpose: rows become movies, columns become users
item_vectors = ratings.T

# Normalize movie vectors
normalized_items = F.normalize(item_vectors, p=2, dim=1)

# Compute item-item cosine similarity
item_similarity = normalized_items @ normalized_items.T

print(f"Item similarity matrix shape: {item_similarity.shape}")

Item similarity matrix shape: torch.Size([3650, 3650])


The resulting matrix tells us how similar each movie is to every other movie, based on how users rated them.

In [7]:
def recommend_movies_item_based(user_id, ratings, item_similarity, movie_titles, top_k=5):
    user_idx = user_id - 1

    # Ratings given by the target user
    user_ratings = ratings[user_idx]

    # Weighted sum of item similarities using the user's existing ratings
    scores = user_ratings @ item_similarity

    # Normalize by total similarity connected to rated items
    rated_mask = user_ratings > 0
    normalization = rated_mask.float() @ item_similarity
    scores = scores / torch.clamp(normalization, min=1e-8)

    # Do not recommend movies the user already rated
    scores[rated_mask] = -1

    top_indices = torch.topk(scores, top_k).indices

    return [movie_titles[i] for i in top_indices]

We can now request recommendations for a user:

In [8]:
movie_titles = user_item.columns.tolist()

recommendations = recommend_movies_item_based(
    user_id=1,
    ratings=ratings,
    item_similarity=item_similarity,
    movie_titles=movie_titles,
    top_k=5
)


print(f"Recommended movies for user_id 1:")
for movie in recommendations:
    print(movie)

Recommended movies for user_id 1:
Buffalo 66 (1998)
Fast Times at Ridgemont High (1982)
Affliction (1997)
Turn It Up (2000)
All the Vermeers in New York (1990)
